In [28]:
import pickle
import ast
import pandas as pd
from collections import Counter

In [2]:
def load_pickle(file):
    """
    load a pickle file into a local variable
    can be a pandas dataframe or dictionary
    """
    pickle_in = open(file, "rb")
    pickle_file = pickle.load(pickle_in)
    pickle_in.close()
    return pickle_file

In [3]:
node_dict = load_pickle('node_dictionary.pickle')
top_paths = load_pickle(f'../results_pearson/network_paths/top_pathways.pickle')

# Negative Controls

In [4]:
n_permutations = 100
my_count = {}
for n in range(n_permutations):
    d = load_pickle(f'../results_pearson/negative_controls/vm/top_pathways_permuted{n}.pickle')
    for val in d.values():
        for el in val.values():
            for l in el:
                if str(l) in my_count:
                    my_count[str(l)] += 1
                else:
                    my_count[str(l)] = 1

Hay un problema con el dict que guarda los top paths: las keys son strings, no listas. Tengo que transformarlo.

In [5]:
rows = []
for key, el in my_count.items():
    key = ast.literal_eval(key)
    rows.append({"path": key, "count": el})

df = pd.DataFrame(rows)

In [6]:
df

,path,count
0,"[APOE_A1, ANGULL01_FDG, MMSE]",13
1,"[APOE_A1, APP, MMSE]",20
2,"[APOE_A1, FRTSUPL01_FDG, MMSE]",9
3,"[APOE_A1, TS_RATIO_ADJ, MMSE]",27
4,"[APOE_A1, MMSE]",72
...,...,...
915152,"[MH12RENA, PTHAND, ADSP_DX]",1
915153,"[MH12RENA, PTHAND, ADSP_MEM]",1
915154,"[MH12RENA, PTHAND, ADSP_EXF]",1
915155,"[MH12RENA, PTHAND, ADSP_LAN]",1


In [7]:
paths = []
for _, value in top_paths.items():
    for _, elements in value.items():
        for el in elements:
            paths.append(el)
paths_permuted = df.path.tolist()

In [36]:
print(len(paths))
print(len(paths_permuted))

30000
915157


In [8]:
not_in_permuted = [x for x in paths if x not in paths_permuted]
len(not_in_permuted)

18342

In [9]:
rows = []
for el in not_in_permuted:
    rows.append({"path": el, "count": 0})
df_append = pd.DataFrame(rows)
df_append

,path,count
0,"[APOE_A1, MH3HEAD, MMSE]",0
1,"[APOE_A1, MH17MALI, MMSE]",0
2,"[APOE_A1, MH5RESP, MMSE]",0
3,"[APOE_A1, CEREB_TCC, MMSE]",0
4,"[APOE_A1, TOTAL_CSF, MMSE]",0
...,...,...
18337,"[MH18SURG, MH10GAST, UW_EF]",0
18338,"[MH18SURG, MH7DERM, UW_EF]",0
18339,"[MH18SURG, MH16SMOK, UW_EF]",0
18340,"[MH18SURG, MH12RENA, UW_EF]",0


In [10]:
# concatenate the dataframes vertically
df = pd.concat([df, df_append], ignore_index=True)
df

,path,count
0,"[APOE_A1, ANGULL01_FDG, MMSE]",13
1,"[APOE_A1, APP, MMSE]",20
2,"[APOE_A1, FRTSUPL01_FDG, MMSE]",9
3,"[APOE_A1, TS_RATIO_ADJ, MMSE]",27
4,"[APOE_A1, MMSE]",72
...,...,...
933494,"[MH18SURG, MH10GAST, UW_EF]",0
933495,"[MH18SURG, MH7DERM, UW_EF]",0
933496,"[MH18SURG, MH16SMOK, UW_EF]",0
933497,"[MH18SURG, MH12RENA, UW_EF]",0


In [13]:
df[df['count'] == 0]

,path,count
915157,"[APOE_A1, MH3HEAD, MMSE]",0
915158,"[APOE_A1, MH17MALI, MMSE]",0
915159,"[APOE_A1, MH5RESP, MMSE]",0
915160,"[APOE_A1, CEREB_TCC, MMSE]",0
915161,"[APOE_A1, TOTAL_CSF, MMSE]",0
...,...,...
933494,"[MH18SURG, MH10GAST, UW_EF]",0
933495,"[MH18SURG, MH7DERM, UW_EF]",0
933496,"[MH18SURG, MH16SMOK, UW_EF]",0
933497,"[MH18SURG, MH12RENA, UW_EF]",0


In [14]:
df[df['count'] <= 1]

,path,count
127,"[APOE_A2, UPplasma_AB42, AXBREATH, MMSE]",1
129,"[APOE_A2, UPplasma_AB42, ST24TS, MMSE]",1
133,"[APOE_A2, UPplasma_AB42, AXBREATH, MOCA]",1
136,"[APOE_A2, UPplasma_AB42, AXCHEST, MOCA]",1
137,"[APOE_A2, UPplasma_AB42, FUSFRML02_FDG, MOCA]",1
...,...,...
933494,"[MH18SURG, MH10GAST, UW_EF]",0
933495,"[MH18SURG, MH7DERM, UW_EF]",0
933496,"[MH18SURG, MH16SMOK, UW_EF]",0
933497,"[MH18SURG, MH12RENA, UW_EF]",0


In [15]:
path_pass = df[df['count'] == 0].path.tolist() # pathways that are present in less than 1% of the permuted pathways

In [16]:
rows = []
for el in path_pass:
    rows.append({"path": el, "source": el[0], "target": el[-1]})
df_pass = pd.DataFrame(rows)
df_pass

,path,source,target
0,"[APOE_A1, MH3HEAD, MMSE]",APOE_A1,MMSE
1,"[APOE_A1, MH17MALI, MMSE]",APOE_A1,MMSE
2,"[APOE_A1, MH5RESP, MMSE]",APOE_A1,MMSE
3,"[APOE_A1, CEREB_TCC, MMSE]",APOE_A1,MMSE
4,"[APOE_A1, TOTAL_CSF, MMSE]",APOE_A1,MMSE
...,...,...,...
18337,"[MH18SURG, MH10GAST, UW_EF]",MH18SURG,UW_EF
18338,"[MH18SURG, MH7DERM, UW_EF]",MH18SURG,UW_EF
18339,"[MH18SURG, MH16SMOK, UW_EF]",MH18SURG,UW_EF
18340,"[MH18SURG, MH12RENA, UW_EF]",MH18SURG,UW_EF


In [17]:
df_pass.to_csv('pass_paths.csv', index=False) # paths que han pasado los negative controls

Paths identified when the input started at
- the genetics
- the molecular
- the PET
- the MRI
- the risk factors

In [ ]:
df_pass = pd.read_csv('pass_paths.csv')

In [43]:
for layer in node_dict.keys():
    df_temp = df_pass[df_pass.source.isin(node_dict[layer].copy())]
    print(f'Total number of paths when the input started in the {layer} layer: {len(df_temp)}')

    #Number of times nodes appear in the paths
    counters = [Counter(lst) for lst in df_temp.path.tolist()] # Create a Counter object for each list
    my_dict = dict(sum(counters, Counter())) # Merge the Counter objects into one dictionary

    for key, values in node_dict.items():
        print('-------', key, '--------------')
        for node, count in my_dict.items():
            if node in values:
                print(node, count)
    print('')

Total number of paths when the input started in the GENETIC layer: 557
------- GENETIC --------------
APOE_A1 90
APOE_A2 84
APOE 82
TOMM40_A1 79
TOMM40_A2 60
PHS 91
CIR 71
------- MOLECULAR --------------
------- PET --------------
MMPLS_FDG 2
------- MRI --------------
CEREB_TCC 40
TOTAL_CSF 40
------- RISKFACTORS --------------
MH3HEAD 83
MH17MALI 70
MH5RESP 84
MH16SMOK 51
MH2NEURL 31
MH13ALLE 43
MH10GAST 48
MH7DERM 60
PTDOBYY 5
------- PHENOTYPE --------------
MMSE 47
MOCA 52
CDR 43
ADSP_DX 44
ADSP_MEM 45
ADSP_EXF 45
ADSP_LAN 45
ADSP_VSP 45
ADAS11 46
ADAS13 46
UW_MEM 51
UW_EF 48

Total number of paths when the input started in the MOLECULAR layer: 1473
------- GENETIC --------------
CIR 6
APOE_A2 22
APOE 15
PHS 5
APOE_A1 1
------- MOLECULAR --------------
EUR_AB42 85
EUR_AB42/40 95
FUJI_AB42 100
FUJI_AB42/40 94
UGOT_PLASMAPTAU 97
TS_RATIO 132
TL 282
TS_RATIO_ADJ 208
BACE 125
UPKelec_TAU 55
UPKelec_PTAU 60
UPplasma_AB42 58
APP 102
UPK_AB42 83
UPK_TAU 83
UPK_PTAU 94
UPKelec_AB42 67
--